In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
base_dir = "/kaggle/input/datasets/anaghabpoojari11/surgbench"
tasks = ["Suturing", "Needle_Passing", "Knot_Tying"]
for task in tasks:
    print(f"\n===== {task} =====")
    task_dir = os.path.join(base_dir, task, task)
    for root, dirs, files in os.walk(task_dir):
        rel_path = os.path.relpath(root, task_dir)
        for f in sorted(files):
            print(os.path.join(rel_path, f))

In [ ]:
import pandas as pd
import glob
import os

TASKS = ['Suturing', 'Knot_Tying', 'Needle_Passing']
BASE = '/kaggle/input/datasets/anaghabpoojari11/surgbench'

rows = []

for task in TASKS:

    # Your dataset has an extra folder with the same name
    task_dir = os.path.join(BASE, task, task)

    trans_dir = os.path.join(task_dir, 'transcriptions')
    video_dir = os.path.join(task_dir, 'video')

    print(f"\nProcessing {task}")
    print("Transcription folder:", trans_dir)
    print("Video folder:", video_dir)

    txt_files = glob.glob(os.path.join(trans_dir, '*.txt'))
    print("Found", len(txt_files), "transcription files")

    for f in txt_files:

        trial_name = os.path.basename(f).replace('.txt', '')

        # Example: Suturing_B001 -> B
        subject = trial_name.split('_')[-1][0]

        video_path = os.path.join(video_dir, trial_name + '_capture2.avi')

        if not os.path.exists(video_path):
            print("Missing video:", video_path)
            continue

        with open(f) as fh:
            for line in fh:

                parts = line.strip().split()

                if len(parts) != 3:
                    continue

                start, end, gesture = parts

                rows.append({
                    'task': task,
                    'trial': trial_name,
                    'subject': subject,
                    'video_path': video_path,
                    'start_frame': int(start),
                    'end_frame': int(end),
                    'gesture': gesture
                })

df = pd.DataFrame(rows)

print("\nTotal rows:", len(df))
print(df.head())

print("\nGesture counts:")
print(df['gesture'].value_counts())

df.to_csv('/kaggle/working/gesture_segments.csv', index=False)

In [ ]:
test_subjects = ['G', 'H']   # held out for testing
train_df = df[~df['subject'].isin(test_subjects)].reset_index(drop=True)
test_df  = df[df['subject'].isin(test_subjects)].reset_index(drop=True)

train_df.to_csv('/kaggle/working/train.csv', index=False)
test_df.to_csv('/kaggle/working/test.csv', index=False)

print(len(train_df), len(test_df))
print(train_df['subject'].unique(), test_df['subject'].unique())

In [ ]:
print(test_df['gesture'].value_counts())
print(train_df['gesture'].value_counts())

In [ ]:
# Remove G10 from all dataframes
df = df[df['gesture'] != 'G10'].reset_index(drop=True)
train_df = train_df[train_df['gesture'] != 'G10'].reset_index(drop=True)
test_df = test_df[test_df['gesture'] != 'G10'].reset_index(drop=True)

In [ ]:
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

GESTURE_LIST = sorted(df['gesture'].unique())          
GESTURE_TO_IDX = {g: i for i, g in enumerate(GESTURE_LIST)}
print(GESTURE_TO_IDX)

class JigsawsGestureDataset(Dataset):
    def __init__(self, dataframe, clip_len=16, resize=(112, 112), train=True):
        self.df = dataframe.reset_index(drop=True)
        self.clip_len = clip_len
        self.resize = resize
        self.train = train

    def __len__(self):
        return len(self.df)

    def _sample_frame_indices(self, start, end):
        total = end - start + 1
        if total <= self.clip_len:
            idxs = np.linspace(start, end, self.clip_len).astype(int)
        else:
            idxs = np.linspace(start, end, self.clip_len).astype(int)
        return idxs

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cap = cv2.VideoCapture(row['video_path'])
        frame_idxs = self._sample_frame_indices(row['start_frame'], row['end_frame'])

        frames = []
        for fi in frame_idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            if not ret:
                frame = np.zeros((self.resize[0], self.resize[1], 3), dtype=np.uint8)
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, self.resize)
            frames.append(frame)
        cap.release()

        clip = np.stack(frames, axis=0).astype(np.float32) / 255.0   # (T, H, W, C)
        clip = torch.from_numpy(clip).permute(3, 0, 1, 2)            # (C, T, H, W)

        mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
        std  = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)
        clip = (clip - mean) / std

        label = GESTURE_TO_IDX[row['gesture']]
        return clip, label

In [ ]:
train_df.to_csv('/kaggle/working/train_final.csv', index=False)
test_df.to_csv('/kaggle/working/test_final.csv', index=False)

In [ ]:
import cv2, numpy as np, os
from tqdm import tqdm

EXTRACT_DIR = '/kaggle/working/clips'
os.makedirs(EXTRACT_DIR, exist_ok=True)

def sample_frame_indices(start, end, clip_len=16):
    return np.linspace(start, end, clip_len).astype(int)

def extract_clip(video_path, start, end, clip_len=16, resize=(112,112)):
    cap = cv2.VideoCapture(video_path)
    idxs = sample_frame_indices(start, end, clip_len)
    frames = []
    for fi in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
        ret, frame = cap.read()
        if not ret:
            frame = np.zeros((resize[0], resize[1], 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, resize)
        frames.append(frame)
    cap.release()
    return np.stack(frames, axis=0)  # (T, H, W, C) uint8

def preprocess_split(dataframe, split_name):
    paths = []
    for i, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
        clip = extract_clip(row['video_path'], row['start_frame'], row['end_frame'])
        out_path = os.path.join(EXTRACT_DIR, f"{split_name}_{i}.npy")
        np.save(out_path, clip)
        paths.append(out_path)
    dataframe = dataframe.copy()
    dataframe['clip_path'] = paths
    return dataframe


In [ ]:
train_df = preprocess_split(train_df, 'train')
test_df = preprocess_split(test_df, 'test')

train_df.to_csv('/kaggle/working/train_final.csv', index=False)
test_df.to_csv('/kaggle/working/test_final.csv', index=False)

In [ ]:
import torch
from torch.utils.data import Dataset

class JigsawsClipDataset(Dataset):
    def __init__(self, dataframe, gesture_to_idx):
        self.df = dataframe.reset_index(drop=True)
        self.gesture_to_idx = gesture_to_idx
        self.mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3,1,1,1)
        self.std  = torch.tensor([0.22803, 0.22145, 0.216989]).view(3,1,1,1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clip = np.load(row['clip_path']).astype(np.float32) / 255.0   # (T,H,W,C)
        clip = torch.from_numpy(clip).permute(3,0,1,2)                # (C,T,H,W)
        clip = (clip - self.mean) / self.std
        label = self.gesture_to_idx[row['gesture']]
        return clip, label

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

train_ds = JigsawsClipDataset(train_df, GESTURE_TO_IDX)
test_ds = JigsawsClipDataset(test_df, GESTURE_TO_IDX)

class_counts = train_df['gesture'].value_counts().to_dict()
sample_weights = train_df['gesture'].map(lambda g: 1.0 / class_counts[g]).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
clip, label = next(iter(train_loader))
print(clip.shape, label.shape, label)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

NUM_CLASSES = len(GESTURE_TO_IDX)   # 13

# Load pretrained on Kinetics-400 -- transfer learning, don't train from scratch
weights = R2Plus1D_18_Weights.KINETICS400_V1
model = r2plus1d_18(weights=weights)

# Replace final classification layer for our 13 gesture classes
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array(sorted(GESTURE_TO_IDX.values()))
y_train = train_df['gesture'].map(GESTURE_TO_IDX).values
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [ ]:
model.eval()
with torch.no_grad():
    clip, label = next(iter(train_loader))
    clip = clip.to(device)
    out = model(clip)
    print(out.shape)   # expect torch.Size([8, 13])

In [ ]:
import time
from sklearn.metrics import f1_score, accuracy_score
import copy

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for clips, labels in loader:
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(clips)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * clips.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for clips, labels in loader:
            clips, labels = clips.to(device), labels.to(device)
            outputs = model(clips)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * clips.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels

In [ ]:
best_f1 = 0.0
best_model_wts = copy.deepcopy(model.state_dict())
history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
           'val_loss': [], 'val_acc': [], 'val_f1': []}

NUM_EPOCHS = 50

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, test_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), '/kaggle/working/best_c2plus1d_gesture.pth')

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"train_loss {train_loss:.3f} acc {train_acc:.3f} f1 {train_f1:.3f} | "
          f"val_loss {val_loss:.3f} acc {val_acc:.3f} f1 {val_f1:.3f} | "
          f"{time.time()-t0:.1f}s")

model.load_state_dict(best_model_wts)
print(f"Best val macro-F1: {best_f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss'])+1)

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs, history['train_f1'], label='train')
axes[1].plot(epochs, history['val_f1'], label='val')
axes[1].set_title('Macro-F1'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=200)
plt.show()

In [ ]:
val_loss, val_acc, val_f1, preds, labels = evaluate(model, test_loader, criterion, device)
print(f"Final best-model val acc: {val_acc:.4f}, macro-F1: {val_f1:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

idx_to_gesture = {v: k for k, v in GESTURE_TO_IDX.items()}
label_names = [idx_to_gesture[i] for i in range(len(GESTURE_TO_IDX))]

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_names, yticklabels=label_names, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix — C(2+1)D Gesture Recognition (JIGSAWS)')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=200)
plt.show()

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(labels, preds, target_names=label_names, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv('/kaggle/working/classification_report.csv')
print(report_df.round(3))

In [ ]:
# Model size and parameter count
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,} ({num_params/1e6:.2f}M)")

# Inference time per clip (single-sample latency)
import time
model.eval()
clip, _ = next(iter(test_loader))
clip_single = clip[0:1].to(device)

with torch.no_grad():
    # warm-up
    for _ in range(5):
        _ = model(clip_single)
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(50):
        _ = model(clip_single)
    torch.cuda.synchronize()
    avg_time = (time.time() - start) / 50

print(f"Avg inference time per clip: {avg_time*1000:.2f} ms ({1/avg_time:.1f} clips/sec)")